In [1]:
!pip install transformers accelerate evaluate jiwer datasets pillow optuna mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 93.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 94.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 86.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
import shutil
import zipfile

import evaluate
import numpy as np
import optuna
import pandas as pd
import torch
from datasets import Dataset
from IPython.display import FileLink, display
from PIL import Image
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    Trainer,
    TrainerCallback,
    TrainingArguments,
    TrOCRProcessor,
    VisionEncoderDecoderModel,
)

print(f"GPUs available: {torch.cuda.device_count()}")

checkpoint_dir = "/kaggle/input/datasets/abhshkgtm/medvision-nlp-checkpoint"
working_dir = "/kaggle/working"

for db_file in ["mlruns.db", "optuna_study.db"]:
    src = os.path.join(checkpoint_dir, db_file)
    dst = os.path.join(working_dir, db_file)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f"Resuming from checkpoint: Restored {db_file}")

os.environ["MLFLOW_EXPERIMENT_NAME"] = "MedVision_Training_Bayesian_Optimization"
os.environ["MLFLOW_TRACKING_URI"] = f"sqlite:///{working_dir}/mlruns.db"
optuna_storage = f"sqlite:///{working_dir}/optuna_study.db"

class AutoBackupCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        with zipfile.ZipFile(f'{working_dir}/training_databases_backup.zip', 'w') as zipf:
            if os.path.exists(f'{working_dir}/mlruns.db'):
                zipf.write(f'{working_dir}/mlruns.db', arcname='mlruns.db')
            if os.path.exists(f'{working_dir}/optuna_study.db'):
                zipf.write(f'{working_dir}/optuna_study.db', arcname='optuna_study.db')


GPUs available: 2
Resuming from checkpoint: Restored mlruns.db
Resuming from checkpoint: Restored optuna_study.db


In [6]:
bert_data_path = "/kaggle/input/datasets/abhshkgtm/medvision-nlp/bio_clinicalbert_dataset.jsonl"
if os.path.exists(bert_data_path):
    df_bert = pd.read_json(bert_data_path, lines=True)
    labels = df_bert['label'].unique().tolist()
    label2id = {l: i for i, l in enumerate(labels)}
    id2label = {i: l for i, l in enumerate(labels)}
    df_bert['label_id'] = df_bert['label'].map(label2id)
    dataset_bert = Dataset.from_pandas(df_bert)
    dataset_bert = dataset_bert.train_test_split(test_size=0.1, seed=42)
    tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
    def tokenize_function(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)
    tokenized_datasets = dataset_bert.map(tokenize_function, batched=True)
    tokenized_datasets = tokenized_datasets.remove_columns(['id', 'source', 'modality', 'text', 'label'])
    tokenized_datasets = tokenized_datasets.rename_column("label_id", "labels")
    tokenized_datasets.set_format("torch")
    import torch.nn as nn
    class_counts = df_bert['label_id'].value_counts().sort_index().values
    total_samples = len(df_bert)
    num_classes = len(labels)
    class_weights = torch.tensor([total_samples / (num_classes * max(c, 1)) for c in class_counts], dtype=torch.float)
else:
    print(f"Dataset not found at {bert_data_path}")


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/8589 [00:00<?, ? examples/s]

Map:   0%|          | 0/955 [00:00<?, ? examples/s]

In [7]:
if os.path.exists(bert_data_path):
    metric = evaluate.load("f1")
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        return metric.compute(predictions=predictions, references=labels, average="macro")

    def monkeypatch_load_state_dict(model):
        original_load = model.load_state_dict
        def new_load(state_dict, strict=True, assign=False):
            new_state_dict = {}
            for k, v in state_dict.items():
                if k.endswith('.beta'):
                    new_state_dict[k.replace('.beta', '.bias')] = v
                elif k.endswith('.gamma'):
                    new_state_dict[k.replace('.gamma', '.weight')] = v
                else:
                    new_state_dict[k] = v
            return original_load(new_state_dict, strict=False, assign=assign)
        model.load_state_dict = new_load

    def model_init_bert():
        model = AutoModelForSequenceClassification.from_pretrained(
            "emilyalsentzer/Bio_ClinicalBERT", 
            num_labels=len(labels), id2label=id2label, label2id=label2id
        )
        monkeypatch_load_state_dict(model)
        return model

    def optuna_hp_space_bert(trial):
        return {
            "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True),
            "weight_decay": trial.suggest_float("weight_decay", 0.001, 0.1, log=True),
            "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [8, 16])
        }

    training_args = TrainingArguments(
        output_dir="/kaggle/working/bert-results",
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        num_train_epochs=15, fp16=True, 
        load_best_model_at_end=True, metric_for_best_model="eval_f1", greater_is_better=True,
        report_to=["mlflow"], label_smoothing_factor=0.1,
        lr_scheduler_type="cosine", warmup_steps=800, gradient_accumulation_steps=2,
    )

    class ImbalancedTrainer(Trainer):
        def __init__(self, *args, class_weights=None, **kwargs):
            super().__init__(*args, **kwargs)
            if class_weights is not None:
                self.class_weights = class_weights.to(self.args.device)
            else:
                self.class_weights = None
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            logits = outputs.logits
            loss_fct = nn.CrossEntropyLoss(weight=self.class_weights) if self.class_weights is not None else nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
            return (loss, outputs) if return_outputs else loss

    trainer = ImbalancedTrainer(
        model=None, model_init=model_init_bert, args=training_args,
        train_dataset=tokenized_datasets["train"], eval_dataset=tokenized_datasets["test"],
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3), AutoBackupCallback()],
        class_weights=class_weights,
    )

    TOTAL_TRIALS_BERT = 10
    existing_study_bert = optuna.create_study(study_name="bio_clinicalBERT_study", storage=optuna_storage, load_if_exists=True, direction="maximize")
    completed_bert = len(existing_study_bert.trials)
    remaining_bert = max(0, TOTAL_TRIALS_BERT - completed_bert)

    if remaining_bert > 0:
        print(f"Resuming Bayesian Optimization for Bio_ClinicalBERT. {remaining_bert} trials remaining out of {TOTAL_TRIALS_BERT}.")
        try:
            best_run = trainer.hyperparameter_search(
                direction="maximize", backend="optuna", hp_space=optuna_hp_space_bert, 
                n_trials=remaining_bert, study_name="bio_clinicalBERT_study",
                storage=optuna_storage, load_if_exists=True
            )
        except KeyboardInterrupt:
            print("Optimization manually interrupted!")
    else:
        print(f"Bio_ClinicalBERT Optimization is already complete ({completed_bert}/{TOTAL_TRIALS_BERT} trials run). Moving directly to weight recovery!")

    try:
        best_trial_bert = existing_study_bert.best_trial
        print(f"Retraining absolute best model (Trial {best_trial_bert.number}) to export .pth weights...")
        for k, v in best_trial_bert.params.items():
            setattr(training_args, k, v)
        final_trainer_bert = ImbalancedTrainer(
            model=model_init_bert(), args=training_args,
            train_dataset=tokenized_datasets["train"], eval_dataset=tokenized_datasets["test"],
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
            class_weights=class_weights,
        )
        final_trainer_bert.train()
        final_trainer_bert.save_model("/kaggle/working/bio_clinicalBERT_model")
        torch.save(final_trainer_bert.model.state_dict(), "/kaggle/working/bio_clinicalBERT_model.pth")
        print("Bio_ClinicalBERT model saved successfully!")
    except Exception as e:
        print(f"Could not complete final retraining: {e}")


pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Conside

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

[I 2026-06-14 12:42:06,781] Using an existing study with name 'bio_clinicalBERT_study' instead of creating a new one.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Bio_ClinicalBERT model saved successfully!


In [3]:
trocr_data_path = "/kaggle/input/datasets/abhshkgtm/medvision-nlp/trocr_dataset.jsonl"
image_base_dir = "/kaggle/input/datasets/abhshkgtm/medvision-nlp/images/images/" 

if os.path.exists(trocr_data_path):
    df_trocr = pd.read_json(trocr_data_path, lines=True)
    dataset_trocr = Dataset.from_pandas(df_trocr)
    dataset_trocr = dataset_trocr.train_test_split(test_size=0.1, seed=42)
    
    processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
    
    def process_trocr_batch(examples):
        pixel_values = []
        for file_path in examples['file_path']:
            filename = os.path.basename(file_path)
            img_path = os.path.join(image_base_dir, filename)
            image = Image.open(img_path).convert("RGB")
            pixel_values.append(processor(image, return_tensors="pt").pixel_values.squeeze())
            
        tokenized = processor.tokenizer(
            examples['text'], padding="max_length", max_length=128, truncation=True
        ).input_ids
        
        labels = [[-100 if token == processor.tokenizer.pad_token_id else token for token in label] for label in tokenized]
        
        # Explicitly build decoder_input_ids by shifting labels right.
        # Prevents DataParallel scatter issue with auto-shift in model.forward()
        decoder_input_ids = []
        for label_seq in tokenized:
            shifted = [processor.tokenizer.cls_token_id] + label_seq[:-1]
            shifted = [processor.tokenizer.pad_token_id if t == processor.tokenizer.pad_token_id else t for t in shifted]
            decoder_input_ids.append(shifted)
        
        return {"pixel_values": pixel_values, "labels": labels, "decoder_input_ids": decoder_input_ids}
        
    encoded_trocr = dataset_trocr.map(process_trocr_batch, batched=True, remove_columns=dataset_trocr["train"].column_names)
    encoded_trocr.set_format(type="torch")
    print(f"TrOCR dataset columns: {encoded_trocr['train'].column_names}")
else:
    print(f"Dataset not found at {trocr_data_path}")


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/932 [00:00<?, ? examples/s]

Map:   0%|          | 0/104 [00:00<?, ? examples/s]

TrOCR dataset columns: ['pixel_values', 'labels', 'decoder_input_ids']


In [4]:
import evaluate
import optuna
import torch
from transformers import EarlyStoppingCallback


# 1. Custom Data Collator for TrOCR
def trocr_data_collator(features):
    # features is a list of dictionaries, one for each example in the batch
    pixel_values = torch.stack([f["pixel_values"].clone().detach() for f in features])
    
    # Extract labels and decoder_input_ids to pad them
    labels = [{"input_ids": f["labels"]} for f in features]
    decoder_input_ids = [{"input_ids": f["decoder_input_ids"]} for f in features]
    
    # Use tokenizer to dynamically pad the text sequences
    labels_padded = processor.tokenizer.pad(labels, padding=True, return_tensors="pt")["input_ids"]
    decoder_input_ids_padded = processor.tokenizer.pad(decoder_input_ids, padding=True, return_tensors="pt")["input_ids"]
    
    # Replace padding token id in labels with -100 so it's ignored in loss calculation
    labels_padded[labels_padded == processor.tokenizer.pad_token_id] = -100
    
    return {
        "pixel_values": pixel_values,
        "labels": labels_padded,
        "decoder_input_ids": decoder_input_ids_padded
    }

if os.path.exists(trocr_data_path):
    cer_metric = evaluate.load("cer")
    
    def compute_cer(pred):
        labels_ids = pred.label_ids
        pred_ids = pred.predictions
        # DataParallel pads gathered predictions with -100 to make lengths match across GPUs
        # We must replace -100 with pad_token_id before decoding to prevent OverflowError
        pred_ids[pred_ids == -100] = processor.tokenizer.pad_token_id
        pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
        
        labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id
        label_str = processor.batch_decode(labels_ids, skip_special_tokens=True)
        cer = cer_metric.compute(predictions=pred_str, references=label_str)
        return {"cer": cer}

    def model_init_trocr():
        model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")
        # Architectural token IDs stay on model.config
        model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
        model.config.pad_token_id = processor.tokenizer.pad_token_id
        model.config.vocab_size = model.config.decoder.vocab_size
        model.config.eos_token_id = processor.tokenizer.sep_token_id
        
        # Generation params MUST go on generation_config in transformers v5.x
        model.generation_config.max_length = 128
        model.generation_config.no_repeat_ngram_size = 3
        model.generation_config.num_beams = 1 # CHANGED FROM 4 TO 1 TO PREVENT DATAPARALLEL DEADLOCK
        
        # FIX: early_stopping and length_penalty removed because they are incompatible with num_beams=1
        
        return model

    def optuna_hp_space_trocr(trial):
        return {
            "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True),
            "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [4, 8])
        }

    training_args_trocr = Seq2SeqTrainingArguments(
        output_dir="/kaggle/working/trocr-results",
        predict_with_generate=True,
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        fp16=True, num_train_epochs=15,
        load_best_model_at_end=True, metric_for_best_model="eval_cer", greater_is_better=False,
        report_to=["mlflow"], label_smoothing_factor=0.1,
        lr_scheduler_type="cosine", warmup_steps=150, gradient_accumulation_steps=2,
        remove_unused_columns=False # Important for multi-modal collators
    )
    
    trainer_trocr = Seq2SeqTrainer(
        model=None, model_init=model_init_trocr,
        processing_class=processor,
        args=training_args_trocr, compute_metrics=compute_cer,
        train_dataset=encoded_trocr["train"], eval_dataset=encoded_trocr["test"],
        data_collator=trocr_data_collator, # FIX: Using our custom multi-modal collator
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3), AutoBackupCallback()],
    )
    
    TOTAL_TRIALS_TROCR = 10
    existing_study_trocr = optuna.create_study(study_name="trocr_study", storage=optuna_storage, load_if_exists=True, direction="minimize")
    completed_trocr = len(existing_study_trocr.trials)
    remaining_trocr = max(0, TOTAL_TRIALS_TROCR - completed_trocr)

    if remaining_trocr > 0:
        print(f"Resuming Bayesian Optimization for TrOCR. {remaining_trocr} trials remaining out of {TOTAL_TRIALS_TROCR}.")
        try:
            best_run_trocr = trainer_trocr.hyperparameter_search(
                direction="minimize", backend="optuna", hp_space=optuna_hp_space_trocr, 
                n_trials=remaining_trocr, study_name="trocr_study",
                storage=optuna_storage, load_if_exists=True
            )
            print("Optimization finished!")
        except KeyboardInterrupt:
            print("Optimization manually interrupted!")
    else:
        print(f"TrOCR Optimization is already complete ({completed_trocr}/{TOTAL_TRIALS_TROCR} trials run). Moving directly to weight recovery!")

    try:
        best_trial_trocr = existing_study_trocr.best_trial
        print(f"Retraining absolute best model (Trial {best_trial_trocr.number}) to export .pth weights...")
        for k, v in best_trial_trocr.params.items():
            setattr(training_args_trocr, k, v)
            
        final_trainer_trocr = Seq2SeqTrainer(
            model=model_init_trocr(), processing_class=processor,
            args=training_args_trocr, compute_metrics=compute_cer,
            train_dataset=encoded_trocr["train"], eval_dataset=encoded_trocr["test"],
            data_collator=trocr_data_collator,  # FIX: Using our custom multi-modal collator
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
        )
        final_trainer_trocr.train()
        final_trainer_trocr.save_model("/kaggle/working/TrOCR_model")
        processor.save_pretrained("/kaggle/working/TrOCR_model")
        torch.save(final_trainer_trocr.model.state_dict(), "/kaggle/working/TrOCR_model.pth")
        print("TrOCR model saved successfully!")
    except Exception as e:
        print(f"Could not complete final retraining: {e}")

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[I 2026-06-14 11:02:18,653] Using an existing study with name 'trocr_study' instead of creating a new one.


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['decoder.output_projection.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrOCR model saved successfully!


In [5]:
import os
import shutil

print("Zipping Final Models and Databases...")

if os.path.exists('/kaggle/working/bio_clinicalBERT_model'):
    shutil.make_archive('/kaggle/working/bio_clinicalBERT_model', 'zip', '/kaggle/working/bio_clinicalBERT_model')
    display(FileLink(r'bio_clinicalBERT_model.zip'))

if os.path.exists('/kaggle/working/TrOCR_model'):
    shutil.make_archive('/kaggle/working/TrOCR_model', 'zip', '/kaggle/working/TrOCR_model')
    display(FileLink(r'TrOCR_model.zip'))

with zipfile.ZipFile('/kaggle/working/training_databases.zip', 'w') as zipf:
    if os.path.exists('/kaggle/working/mlruns.db'):
        zipf.write('/kaggle/working/mlruns.db', arcname='mlruns.db')
    if os.path.exists('/kaggle/working/optuna_study.db'):
        zipf.write('/kaggle/working/optuna_study.db', arcname='optuna_study.db')

if os.path.exists('/kaggle/working/training_databases.zip'):
    display(FileLink(r'training_databases.zip'))

if os.path.exists('/kaggle/working/training_databases_backup.zip'):
    print("Backup also available:")
    display(FileLink(r'training_databases_backup.zip'))

print("All files zipped! Click links above to download.")


Zipping Final Models and Databases...


/kaggle/working/TrOCR_model.zip

/kaggle/working/training_databases.zip

All files zipped! Click links above to download.
